In [ ]:
!pip install evaluate rouge_score nltk pycocotools transformers datasets --quiet

import os
import json
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
from pycocotools.coco import COCO
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import autocast, GradScaler
from transformers import VisionEncoderDecoderModel, AutoTokenizer
import evaluate
from google.colab import drive

def clean_caption(text):
    """إزالة التكرار المتتالي للكلمات لتحسين التقييم"""
    words = text.split()
    cleaned = []
    for w in words:
        if len(cleaned) < 2 or not (w == cleaned[-1] == cleaned[-2]):
            cleaned.append(w)
    return " ".join(cleaned)

drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/coco_checkpoints"
os.makedirs(save_dir, exist_ok=True)

data_root = "/content/coco_data"
image_folder = os.path.join(data_root, "images")
annotations_folder = os.path.join(data_root, "annotations")
os.makedirs(image_folder, exist_ok=True)
os.makedirs(annotations_folder, exist_ok=True)

if not os.path.exists(os.path.join(image_folder, "train2017")):
    print("Downloading COCO train/val + annotations...")
    !wget http://images.cocodataset.org/zips/train2017.zip -P {image_folder}
    !unzip -q {image_folder}/train2017.zip -d {image_folder}
    !wget http://images.cocodataset.org/zips/val2017.zip -P {image_folder}
    !unzip -q {image_folder}/val2017.zip -d {image_folder}
    !wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P {annotations_folder}
    !unzip -q {annotations_folder}/annotations_trainval2017.zip -d {annotations_folder}
    !rm {image_folder}/train2017.zip {image_folder}/val2017.zip
    !rm {annotations_folder}/annotations_trainval2017.zip

train_image_path = os.path.join(image_folder, "train2017")
val_image_path = os.path.join(image_folder, "val2017")
train_ann_file = os.path.join(annotations_folder, "annotations/captions_train2017.json")
val_ann_file = os.path.join(annotations_folder, "annotations/captions_val2017.json")
coco_train = COCO(train_ann_file)
coco_val = COCO(val_ann_file)

TRAIN_LIMIT = 50000
VAL_LIMIT = 2000
train_img_ids = coco_train.getImgIds()[:TRAIN_LIMIT]
val_img_ids = coco_val.getImgIds()[:VAL_LIMIT]


class CocoCaptionDataset(Dataset):
    def __init__(self, coco, img_ids, image_root, tokenizer, max_length=32):
        self.coco = coco
        self.img_ids = img_ids
        self.image_root = image_root
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.transform = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor()
        ])
    def __len__(self):
        return len(self.img_ids)
    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.image_root, img_info['file_name'])
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        caption = random.choice(anns)["caption"]

        encoding = self.tokenizer(
            caption, max_length=self.max_length, truncation=True,
            padding="max_length", return_tensors="pt"
        )
        labels = encoding.input_ids.squeeze()
        attention_mask = encoding.attention_mask.squeeze()
        return image, labels, attention_mask, caption

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "nlpconnect/vit-gpt2-image-captioning"
checkpoint_path = "/content/drive/MyDrive/coco_checkpoints"
state_file = os.path.join(checkpoint_path, "training_state.json")

if os.path.exists(os.path.join(checkpoint_path, "model.safetensors")):
    print("🔄 استئناف التدريب من آخر Checkpoint...")
    model = VisionEncoderDecoderModel.from_pretrained(checkpoint_path)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    if os.path.exists(state_file):
        with open(state_file, "r") as f:
            state = json.load(f)
        start_epoch = state.get("last_epoch", 0) + 1
        best_val_loss = state.get("best_val_loss", float("inf"))
        train_losses = state.get("train_losses", [])
        val_losses = state.get("val_losses", [])
        bleu_scores = state.get("bleu_scores", [])
        rouge_scores = state.get("rouge_scores", [])
        meteor_scores = state.get("meteor_scores", [])
        accuracies = state.get("accuracies", [])
    else:
        start_epoch = 1
        best_val_loss = float("inf")
        train_losses, val_losses, bleu_scores, rouge_scores, meteor_scores, accuracies = [], [], [], [], [], []
else:
    print("⬇️ تحميل النموذج من HuggingFace...")
    model = VisionEncoderDecoderModel.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    start_epoch = 1
    best_val_loss = float("inf")
    train_losses, val_losses, bleu_scores, rouge_scores, meteor_scores, accuracies = [], [], [], [], [], []
    print("🚀 بدء التدريب من الصفر")

model.config.encoder.hidden_dropout_prob = 0.1
model.config.encoder.attention_probs_dropout_prob = 0.1
model.config.decoder.attn_pdrop = 0.1
model.config.decoder.resid_pdrop = 0.1
model.config.decoder.embd_pdrop = 0.1
model = model.to(DEVICE)

BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
train_dataset = CocoCaptionDataset(coco_train, train_img_ids, train_image_path, tokenizer)
val_dataset = CocoCaptionDataset(coco_val, val_img_ids, val_image_path, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-6)#2e-6


bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

NUM_EPOCHS = 6 # Define NUM_EPOCHS before it is used
from torch.optim.lr_scheduler import CosineAnnealingLR

# إنشاء Scheduler (تغير LR بشكل تدريجي)
scheduler = CosineAnnealingLR(optimizer, T_max=len(train_loader) * NUM_EPOCHS)

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    print(f"\n🎯 بدء Epoch {epoch}/{NUM_EPOCHS}")
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    for step, (images, labels, attention_masks, _) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} - Train", leave=False)):
        images, labels, attention_masks = images.to(DEVICE), labels.to(DEVICE), attention_masks.to(DEVICE)
        with autocast():
            outputs = model(pixel_values=images, labels=labels, attention_mask=attention_masks)
            loss = outputs.loss
            if torch.isnan(loss):
                optimizer.zero_grad()
                continue
            scaler.scale(loss).backward()
            if (step+1) % GRAD_ACCUM_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
            total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    current_lr = scheduler.get_last_lr()[0]
    print(f"📉 Learning Rate الحالي: {current_lr:.8f}")

    model.eval()
    total_val_loss = 0
    predictions, references = [], []
    sample_examples = []
    overlap_sum, total_ref_tokens = 0, 0
    MAX_LEN = 32#25
    with torch.no_grad():
        for images, labels, attention_masks, caps in tqdm(val_loader, desc="Evaluating", leave=False):
            images, labels, attention_masks = images.to(DEVICE), labels.to(DEVICE), attention_masks.to(DEVICE)
            outputs = model(pixel_values=images, labels=labels, attention_mask=attention_masks)
            total_val_loss += outputs.loss.item()
            gen_ids = model.generate(
    pixel_values=images,
    attention_mask=attention_masks,
    max_length=MAX_LEN,
    num_beams=5,               # ← تفعيل Beam Search
    no_repeat_ngram_size=3,
    repetition_penalty=1.5,
    length_penalty=1.0,        # يمكنك تعديلها لضبط طول الجملة
    early_stopping=True
)


            gen_texts = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
            gen_texts = [clean_caption(t) for t in gen_texts]
            predictions.extend(gen_texts)
            references.extend([[c] for c in caps])
            for pred, ref in zip(gen_texts, caps):
                pred_tokens = set(pred.lower().split())
                ref_tokens = set(ref.lower().split())
                overlap_sum += len(pred_tokens.intersection(ref_tokens))
                total_ref_tokens += len(ref_tokens)
            if len(sample_examples) < 5:
                for pred, ref in zip(gen_texts, caps):
                    sample_examples.append((ref, pred))
                    if len(sample_examples) >= 5:
                        break

    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    bleu_score = bleu.compute(predictions=predictions, references=references)["bleu"]
    rouge_score = rouge.compute(predictions=predictions, references=[r[0] for r in references])
    meteor_score = meteor.compute(predictions=predictions, references=[r[0] for r in references])
    accuracy = overlap_sum / total_ref_tokens if total_ref_tokens > 0 else 0

    bleu_scores.append(bleu_score)
    rouge_scores.append(rouge_score["rougeL"])
    meteor_scores.append(meteor_score["meteor"])
    accuracies.append(accuracy)

    print(f"\n📊 Epoch {epoch}/{NUM_EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"BLEU: {bleu_score:.4f} | ROUGE-L: {rouge_score['rougeL']:.4f} | METEOR: {meteor_score['meteor']:.4f} | Accuracy: {accuracy:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained(checkpoint_path)
        tokenizer.save_pretrained(checkpoint_path)
        with open(state_file, "w") as f:
            json.dump({
                "last_epoch": epoch,
                "best_val_loss": best_val_loss,
                "train_losses": train_losses,
                "val_losses": val_losses,
                "bleu_scores": bleu_scores,
                "rouge_scores": rouge_scores,
                "meteor_scores": meteor_scores,
                "accuracies": accuracies
            }, f)
        print(f"✅ أفضل نموذج حتى الآن — تم الحفظ في {checkpoint_path}")

    print("\n📝 أمثلة من الـ Validation:")
    for ref, pred in sample_examples:
        print(f"🟢 المرجع: {ref}")
        print(f"🔵 المولّد: {pred}\n")

rand_idx = random.randint(0, len(gen_texts) - 1)


sample_image = images[rand_idx].cpu().permute(1, 2, 0)  
sample_ref = caps[rand_idx]
sample_pred = gen_texts[rand_idx]

plt.figure(figsize=(5, 5))
plt.imshow(sample_image)
plt.axis("off")
plt.title("📷 صورة من الـ Validation")
plt.show()

print(f"🟢 المرجع: {sample_ref}")
print(f"🔵 المولّد: {sample_pred}")

# ================== رسم المنحنيات ==================
plt.figure(figsize=(8,6))
plt.plot(train_losses, label="Train Loss", color="red")
plt.plot(val_losses, label="Val Loss", color="orange")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves")
plt.legend()
plt.show()
بز
plt.figure(figsize=(8,6))
plt.plot(bleu_scores, label="BLEU", color="blue")
plt.xlabel("Epoch")
plt.ylabel("BLEU")
plt.title("BLEU Score")
plt.legend()
plt.show()

plt.figure(figsize=(8,6))
plt.plot(rouge_scores, label="ROUGE-L", color="green")
plt.plot(meteor_scores, label="METEOR", color="purple")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("ROUGE-L & METEOR Scores")
plt.legend()
plt.show()

plt.figure(figsize=(8,6))
plt.plot(accuracies, label="Accuracy", color="teal")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()
plt.show()

In [ ]:
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from transformers import VisionEncoderDecoderModel, AutoTokenizer
import evaluate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = "/content/drive/MyDrive/coco_checkpoints"

print("🔄 تحميل النموذج من الـ checkpoint...")
model = VisionEncoderDecoderModel.from_pretrained(checkpoint_path).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model.eval()

test_image_folder = "/content/drive/MyDrive/images2"
test_annotations = "/content/drive/MyDrive/output.json/annotations_coco_format.json"

with open(test_annotations, "r") as f:
    test_data = json.load(f)

class TestDataset(Dataset):
    def __init__(self, data, image_root, tokenizer, max_length=32):
        # فلترة الصور التي لا تحتوي على كابتشن
        self.images = [img for img in data["images"]
                       if any(ann["image_id"] == img["id"] for ann in data["annotations"])]
        self.annotations = data["annotations"]
        self.image_root = image_root
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.transform = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info["id"]
        img_path = os.path.join(self.image_root, img_info["file_name"])
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        anns = [ann for ann in self.annotations if ann["image_id"] == img_id]
        captions = [ann["caption"] for ann in anns]

        return image, captions

BATCH_SIZE = 8
test_dataset = TestDataset(test_data, test_image_folder, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

model.eval()
predictions, references = [], []
sample_examples = []
overlap_sum, total_ref_tokens = 0, 0

with torch.no_grad():
    for images, caps in tqdm(test_loader, desc="Evaluating on Test Set", leave=False):
        images = images.to(DEVICE)

        gen_ids = model.generate(
            images,
            max_length=32,
            num_beams=5,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            early_stopping=True
        )
        gen_texts = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

        for pred, ref_list in zip(gen_texts, caps):
            predictions.append(pred)
            references.append(ref_list) # references should be a list of lists for evaluation metrics


bleu_score = bleu.compute(predictions=predictions, references=references)["bleu"]
rouge_score = rouge.compute(predictions=predictions, references=[r[0] for r in references]) # Rouge expects a list of strings for references
meteor_score = meteor.compute(predictions=predictions, references=[r[0] for r in references]) # Meteor expects a list of strings for references


overlap_sum, total_ref_tokens = 0, 0
# For manual accuracy calculation, we can use the first reference caption if multiple exist
for pred, ref_list in zip(predictions, references):
    if ref_list: # Check if there is at least one reference caption
        ref = ref_list[0] # Use the first reference caption
        pred_tokens = set(pred.lower().split())
        ref_tokens = set(ref.lower().split())
        overlap_sum += len(pred_tokens.intersection(ref_tokens))
        total_ref_tokens += len(ref_tokens)

accuracy = overlap_sum / total_ref_tokens if total_ref_tokens > 0 else 0


print("\n📊 نتائج الـ Test:")
print(f"BLEU: {bleu_score:.4f}")
print(f"ROUGE-L: {rouge_score['rougeL']:.4f}")
print(f"METEOR: {meteor_score['meteor']:.4f}")
print(f"Accuracy: {accuracy:.4f}")

print("\n📝 أمثلة من الـ Test:")
# Displaying the first reference caption for examples
for ref_list, pred in zip(references[:5], predictions[:5]):
    ref = ref_list[0] if ref_list else "No reference available"
    print(f"🟢 المرجع: {ref}")
    print(f"🔵 المولّد: {pred}\n")

In [ ]:
import os # Import os here
from PIL import Image
import torch
import matplotlib.pyplot as plt
from torchvision import transforms # Import transforms here

from transformers import VisionEncoderDecoderModel, AutoTokenizer

def clean_caption(text):
    """إزالة التكرار المتتالي للكلمات لتحسين التقييم"""
    words = text.split()
    cleaned = []
    for w in words:
        if len(cleaned) < 2 or not (w == cleaned[-1] == cleaned[-2]):
            cleaned.append(w)
    return " ".join(cleaned)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "nlpconnect/vit-gpt2-image-captioning"
checkpoint_path = "/content/drive/MyDrive/coco_checkpoints"

if os.path.exists(os.path.join(checkpoint_path, "model.safetensors")):
    print("🔄 الحصول على حالة النموذج من Checkpoint...")
    model = VisionEncoderDecoderModel.from_pretrained(checkpoint_path)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
else:
    print("⬇️ تحميل النموذج من HuggingFace...")
    model = VisionEncoderDecoderModel.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

model = model.to(DEVICE)



#image_path = "/content/drive/MyDrive/OIP.jpg"
image_path = "/content/drive/MyDrive/OIP (1) - Copy.jpg"
#image_path = "/content/drive/MyDrive/تنزيل.webp"

image = Image.open(image_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])
image_tensor = transform(image).unsqueeze(0).to(DEVICE)

model.eval()
with torch.no_grad():
    gen_ids = model.generate(
        pixel_values=image_tensor,
        max_length=32,
        num_beams=5,
        no_repeat_ngram_size=3,
        repetition_penalty=1.5,
        length_penalty=1.0,
        early_stopping=True
    )

generated_caption = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
generated_caption = clean_caption(generated_caption)

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.axis("off")
plt.title(f" generator caption:\n{generated_caption}", fontsize=12)
plt.show()